In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    if 'Daily Living' in dirs:
        print("FOUND:", os.path.join(root, 'Daily Living'))
    # also print top-level structure for visibility
    if root.count(os.sep) <= 3:
        print(root)

/kaggle/input
/kaggle/input/datasets
FOUND: /kaggle/input/datasets/nazmulmridha680/dataset-for-fall-and-activity-detection/Generated_Images_Mosaic/Mosaic_ADL_2x2/Daily Living
FOUND: /kaggle/input/datasets/nazmulmridha680/dataset-for-fall-and-activity-detection/Generated_Images_Mosaic/Mosaic_PatchCore_2x2/Daily Living


In [2]:
"""Patchcore dataset organizer · PY
===================================================================================================
PATCHCORE DATASET ORGANIZER -- SisFall Mosaic_PatchCore_2x2 -> Per-Activity PatchCore-Ready Folders
===================================================================================================
This does NOT regenerate any images. Your existing SisFall_RGB_Mosaic_Pipeline_FINAL_v5.py already
produced everything needed (verified by direct code inspection: it processes BOTH Daily Living and
Fall files, and generates Mosaic_PatchCore_2x2 for both). This script REORGANIZES those existing
images into the folder structure one-class anomaly detection (PatchCore) expects:

    PatchCore_Ready/
      Walking/
        train/good/*.png       <- normal Walking images ONLY (memory bank reference set)
        test/good/*.png        <- held-out normal Walking images (different subjects than train)
        test/anomaly/*.png     <- Fall images from F01,F02,F03,F04,F06,F07 (walking-related falls)
      Jogging/
        train/good/*.png
        test/good/*.png
        test/anomaly/*.png     <- F05
      Sitting/
        train/good/*.png
        test/good/*.png
        test/anomaly/*.png     <- F10,F11,F12 (sit-down transition) + F13,F14,F15 (already-seated faint)
      Standing_GettingUp/
        train/good/*.png       <- Collapse_Into_Chair (closest ADL match to a get-up motion)
        test/good/*.png
        test/anomaly/*.png     <- F08,F09 (getting-up falls)

WHY THIS MAPPING (grounded in SisFall's own official fall-type documentation, not a guess):
    F01 Fall forward while walking caused by a slip          -> Walking
    F02 Fall backward while walking caused by a slip          -> Walking
    F03 Lateral fall while walking caused by a slip           -> Walking
    F04 Fall forward while walking caused by a trip           -> Walking
    F05 Fall forward while jogging caused by a trip           -> Jogging
    F06 Vertical fall while walking caused by fainting        -> Walking
    F07 Fall while walking (hands on table), caused by fainting -> Walking
    F08 Fall forward when trying to get up                    -> Standing_GettingUp
    F09 Lateral fall when trying to get up                    -> Standing_GettingUp
    F10 Fall forward when trying to sit down                  -> Sitting
    F11 Fall backward when trying to sit down                 -> Sitting
    F12 Lateral fall when trying to sit down                  -> Sitting
    F13 Fall forward while sitting, fainting/falling asleep    -> Sitting
    F14 Fall backward while sitting, fainting/falling asleep   -> Sitting
    F15 Lateral fall while sitting, fainting/falling asleep    -> Sitting

WHY SUBJECT-WISE SPLIT FOR THE "GOOD" IMAGES SPECIFICALLY:
    train/good and test/good must come from DIFFERENT subjects, for the same leakage reason as
    the ADL classifier -- otherwise "test/good" could partly just be recognizing a person the
    memory bank already saw. Fall images go ENTIRELY into test/anomaly regardless of subject --
    PatchCore never trains on anomalies (that's the whole point of one-class detection), so there's
    no leakage concern there, and using every subject's fall trials maximizes your ability to
    measure true recall (per the trial-level recall discussion -- never miss a fall).

Files are COPIED, not moved -- your original Mosaic_PatchCore_2x2 folder is left untouched.
===================================================================================================
"""

import os
import re
import glob
import shutil
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------------------
# 0. CONFIG
# ---------------------------------------------------------------------------------------
SEED = 42
TRAIN_GOOD_FRACTION = 0.80   # fraction of "good" SUBJECTS (not images) used for train/good

def find_patchcore_root():
    candidates = glob.glob("/kaggle/input/**/Mosaic_PatchCore_2x2", recursive=True)
    if not candidates:
        raise FileNotFoundError(
            "Could not find 'Mosaic_PatchCore_2x2' under /kaggle/input.\n"
            "Make sure your original mosaic dataset is attached to this notebook."
        )
    return candidates[0]

PATCHCORE_ROOT = find_patchcore_root()
print(f"[INFO] Found Mosaic_PatchCore_2x2 at: {PATCHCORE_ROOT}")

OUTPUT_ROOT = "/kaggle/working/PatchCore_Ready"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ---------------------------------------------------------------------------------------
# 1. ACTIVITY GROUPS -- which Daily Living folders feed each PatchCore model's "good" set
# ---------------------------------------------------------------------------------------
ACTIVITY_GROUPS = {
    "Walking": ["Walking_Slow", "Walking_Fast"],
    "Jogging": ["Jogging_Slow", "Jogging_Fast"],
    "Sitting": ["Sit_HalfHeight_Slow", "Sit_HalfHeight_Fast", "Sit_LowHeight_Slow", "Sit_LowHeight_Fast"],
    "Standing_GettingUp": ["Collapse_Into_Chair"],
}

# ---------------------------------------------------------------------------------------
# 2. FALL-CODE -> ACTIVITY MAPPING -- grounded in SisFall's official F01-F15 descriptions
# ---------------------------------------------------------------------------------------
FALL_CODE_TO_ACTIVITY = {
    "F01": "Walking", "F02": "Walking", "F03": "Walking", "F04": "Walking",
    "F06": "Walking", "F07": "Walking",
    "F05": "Jogging",
    "F08": "Standing_GettingUp", "F09": "Standing_GettingUp",
    "F10": "Sitting", "F11": "Sitting", "F12": "Sitting",
    "F13": "Sitting", "F14": "Sitting", "F15": "Sitting",
}

# ---------------------------------------------------------------------------------------
# 3. BUILD TRAIN/GOOD, TEST/GOOD (subject-wise split within each activity group)
# ---------------------------------------------------------------------------------------
def collect_good_images(activity_group_name, subfolders):
    """Returns a DataFrame of (filepath, subject, sub_activity) for every image across
    the group's Daily Living subfolders. 'sub_activity' (e.g. 'Sit_HalfHeight_Slow')
    is preserved -- NOT just the merged group name -- so the trainer can later build
    one balanced memory bank per original sub-activity instead of one pooled bank per
    group. This matters because a pooled 'Sitting' bank (4 very different sub-activities
    merged) risks the coreset algorithm under-representing whichever sub-activity has
    less/less-diverse data, purely by chance -- tracking sub_activity here lets the
    trainer guarantee every sub-activity gets fair, guaranteed representation."""
    records = []
    for subfolder in subfolders:
        class_dir = os.path.join(PATCHCORE_ROOT, "Daily Living", subfolder)
        if not os.path.isdir(class_dir):
            print(f"[WARN] Missing folder for '{subfolder}' -- skipping.")
            continue
        for subject in sorted(os.listdir(class_dir)):
            subject_path = os.path.join(class_dir, subject)
            if not os.path.isdir(subject_path):
                continue
            for fpath in glob.glob(os.path.join(subject_path, "*.png")):
                records.append({"filepath": fpath, "subject": subject, "sub_activity": subfolder})
    return pd.DataFrame(records)

def window_wise_good_split(good_df, train_frac=TRAIN_GOOD_FRACTION, seed=SEED):
    """Window-wise split (matches your ADL classifier's methodology): all subjects'
    'good' images pooled together, shuffled at the individual-image level, then
    divided train/test by that fraction -- regardless of which subject each image
    came from. Every subject can appear in both train/good and test/good."""
    idx = good_df.index.to_numpy().copy()
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    n_train = int(round(len(idx) * train_frac))
    train_df = good_df.loc[idx[:n_train]]
    test_df = good_df.loc[idx[n_train:]]
    train_subjects = sorted(train_df["subject"].unique())
    test_subjects = sorted(test_df["subject"].unique())
    return train_df, test_df, train_subjects, test_subjects

# ---------------------------------------------------------------------------------------
# 4. COLLECT FALL (ANOMALY) IMAGES, ROUTED BY FALL CODE
# ---------------------------------------------------------------------------------------
def collect_fall_images_for_activity(activity_group_name):
    fall_dir = os.path.join(PATCHCORE_ROOT, "Fall")
    if not os.path.isdir(fall_dir):
        print(f"[WARN] No 'Fall' folder found at {fall_dir}")
        return pd.DataFrame(columns=["filepath", "subject", "fall_code"])
    records = []
    for subject in sorted(os.listdir(fall_dir)):
        subject_path = os.path.join(fall_dir, subject)
        if not os.path.isdir(subject_path):
            continue
        for fpath in glob.glob(os.path.join(subject_path, "*.png")):
            fname = os.path.basename(fpath)
            m = re.match(r"^(F\d{2})_", fname)
            if not m:
                continue
            fall_code = m.group(1)
            if FALL_CODE_TO_ACTIVITY.get(fall_code) == activity_group_name:
                records.append({"filepath": fpath, "subject": subject, "fall_code": fall_code})
    return pd.DataFrame(records)

# ---------------------------------------------------------------------------------------
# 5. RUN THE ORGANIZATION -- copy files into the PatchCore-ready structure
# ---------------------------------------------------------------------------------------
def copy_files(df, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    for _, row in df.iterrows():
        shutil.copy2(row["filepath"], os.path.join(dest_dir, os.path.basename(row["filepath"])))

def copy_files_by_sub_activity(df, dest_dir):
    """Like copy_files, but places each image into a dest_dir/{sub_activity}/ subfolder
    instead of one flat folder -- this is what lets the trainer build one memory bank
    per original sub-activity (e.g. dest_dir/Sit_HalfHeight_Slow/, dest_dir/Sit_HalfHeight_Fast/,
    ...) instead of a single pooled bank that risks under-representing some sub-activities."""
    for _, row in df.iterrows():
        sub_dir = os.path.join(dest_dir, row["sub_activity"])
        os.makedirs(sub_dir, exist_ok=True)
        shutil.copy2(row["filepath"], os.path.join(sub_dir, os.path.basename(row["filepath"])))

summary_rows = []
for activity_group_name, subfolders in ACTIVITY_GROUPS.items():
    print(f"\n{'='*70}\n{activity_group_name}\n{'='*70}")

    good_df = collect_good_images(activity_group_name, subfolders)
    if len(good_df) == 0:
        print(f"[WARN] No 'good' images found for {activity_group_name} -- skipping this group.")
        continue
    train_good_df, test_good_df, train_subjects, test_subjects = window_wise_good_split(good_df)

    fall_df = collect_fall_images_for_activity(activity_group_name)

    group_root = os.path.join(OUTPUT_ROOT, activity_group_name)
    copy_files_by_sub_activity(train_good_df, os.path.join(group_root, "train", "good"))
    copy_files(test_good_df, os.path.join(group_root, "test", "good"))
    copy_files(fall_df, os.path.join(group_root, "test", "anomaly"))

    print(f"  Source Daily Living folders : {subfolders}")
    print(f"  train/good subjects ({len(train_subjects)}): {train_subjects}")
    print(f"  test/good subjects  ({len(test_subjects)}): {test_subjects}")
    print(f"  train/good images   : {len(train_good_df)}")
    print(f"  test/good images    : {len(test_good_df)}")
    print(f"  test/anomaly images : {len(fall_df)} "
          f"(fall codes: {sorted(fall_df['fall_code'].unique()) if len(fall_df) else 'NONE FOUND'})")

    summary_rows.append({
        "activity_group": activity_group_name,
        "train_good": len(train_good_df),
        "test_good": len(test_good_df),
        "test_anomaly": len(fall_df),
        "fall_codes_included": ",".join(sorted(fall_df["fall_code"].unique())) if len(fall_df) else "",
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 70 + "\nORGANIZATION SUMMARY\n" + "=" * 70)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(OUTPUT_ROOT, "organization_summary.csv"), index=False)

print(f"\n[DONE] PatchCore-ready dataset built at: {OUTPUT_ROOT}")
print("[NOTE] Original Mosaic_PatchCore_2x2 folder is untouched -- files were COPIED, not moved.")
print("[NEXT STEP] Train one PatchCore model per activity folder above, in priority order:")
print("            Walking (6/15 fall types) -> Sitting (6/15) -> Standing_GettingUp (2/15) -> Jogging (1/15)")

[INFO] Found Mosaic_PatchCore_2x2 at: /kaggle/input/datasets/nazmulmridha680/dataset-for-fall-and-activity-detection/Generated_Images_Mosaic/Mosaic_PatchCore_2x2

Walking
  Source Daily Living folders : ['Walking_Slow', 'Walking_Fast']
  train/good subjects (38): ['SA01', 'SA02', 'SA03', 'SA04', 'SA05', 'SA06', 'SA07', 'SA08', 'SA09', 'SA10', 'SA11', 'SA12', 'SA13', 'SA14', 'SA15', 'SA16', 'SA17', 'SA18', 'SA19', 'SA20', 'SA21', 'SA22', 'SA23', 'SE01', 'SE02', 'SE03', 'SE04', 'SE05', 'SE06', 'SE07', 'SE08', 'SE09', 'SE10', 'SE11', 'SE12', 'SE13', 'SE14', 'SE15']
  test/good subjects  (38): ['SA01', 'SA02', 'SA03', 'SA04', 'SA05', 'SA06', 'SA07', 'SA08', 'SA09', 'SA10', 'SA11', 'SA12', 'SA13', 'SA14', 'SA15', 'SA16', 'SA17', 'SA18', 'SA19', 'SA20', 'SA21', 'SA22', 'SA23', 'SE01', 'SE02', 'SE03', 'SE04', 'SE05', 'SE06', 'SE07', 'SE08', 'SE09', 'SE10', 'SE11', 'SE12', 'SE13', 'SE14', 'SE15']
  train/good images   : 2387
  test/good images    : 597
  test/anomaly images : 3529 (fall codes:

In [3]:
"""Refine anomaly windows · PY
===================================================================================================
ANOMALY WINDOW REFINER -- Fixes the mislabeled pre-fall/post-fall window problem
FINAL VERSION -- adds a third category: prefall_context (genuinely normal, kept separately)
===================================================================================================
Run this AFTER patchcore_dataset_organizer.py and BEFORE train_patchcore_models.py.

THE PROBLEM THIS FIXES (confirmed by your own results):
A 15-second fall trial with 50% overlap produces ~5 windows, but only 1-2 of them actually
contain the impact. The v1 version of this script excluded ALL non-impact windows equally.
This version is more precise: it recognizes that windows well BEFORE the impact are genuinely
normal activity -- not just "not confidently anomalous" -- and routes them to a NEW,
separate test category (test/prefall_context) instead of discarding them.

THREE-WAY CLASSIFICATION PER TRIAL (windows sorted by their time position in the trial):
  1. KEPT AS ANOMALY: the TOP_K_WINDOWS_PER_TRIAL highest kurtogram-intensity windows --
     these are your genuine impact windows.
  2. PREFALL_CONTEXT (NEW): windows that are EARLY -- at least PREFALL_BUFFER_WINDOWS
     positions before the earliest kept impact window. These are confidently normal,
     un-ambiguous activity and get saved to test/prefall_context/ as their own evaluation
     set -- this specifically measures "does the model stay calm in the moments right
     before a fall happens", which is more realistic and more informative than lumping
     these in with your main Daily-Living test/good set.
  3. EXCLUDED (ambiguous or post-fall): windows immediately adjacent to the impact window(s)
     (could already contain the onset of a stumble/loss of balance -- too ambiguous to
     confidently label either way) AND all windows AFTER the impact (person is down --
     a genuinely different physical state your Daily-Living data doesn't represent, so
     it's neither confidently "normal" nor "anomaly" in the sense this model was trained
     to detect). These are moved out, not deleted, same as before.

This operates on the ALREADY-ORGANIZED PatchCore_Ready/{Activity}/test/anomaly folders --
no need to re-run the organizer or touch raw sensor files.
===================================================================================================
"""

import os
import re
import glob
import shutil
import numpy as np
import pandas as pd
from PIL import Image

# ---------------------------------------------------------------------------------------
# 0. CONFIG
# ---------------------------------------------------------------------------------------
TOP_K_WINDOWS_PER_TRIAL = 2       # keep the K highest-kurtogram-intensity windows per trial
                                    # as genuine anomaly labels (unchanged from v1).
PREFALL_BUFFER_WINDOWS = 1         # windows within this many positions BEFORE the earliest
                                    # kept impact window are treated as ambiguous (excluded,
                                    # not used anywhere) rather than confidently normal --
                                    # a stumble/loss-of-balance onset could already be
                                    # starting here. Windows FURTHER before that buffer go
                                    # to prefall_context as confidently normal.

# Mosaic_PatchCore_2x2 geometry (verified against the actual generation pipeline)
CELL_SIZE = 224
GUTTER_PX = 8
KURTOGRAM_X0 = CELL_SIZE + GUTTER_PX   # 232 -- kurtogram is the RIGHT column
KURTOGRAM_X1 = KURTOGRAM_X0 + CELL_SIZE  # 456

def find_patchcore_ready_root():
    candidates = glob.glob("/kaggle/working/PatchCore_Ready") + \
                 glob.glob("/kaggle/input/**/PatchCore_Ready", recursive=True)
    if not candidates:
        raise FileNotFoundError("Could not find 'PatchCore_Ready' -- run the organizer script first.")
    return candidates[0]

PATCHCORE_READY_ROOT = find_patchcore_ready_root()
print(f"[INFO] Refining anomaly labels in: {PATCHCORE_READY_ROOT}")

ACTIVITY_GROUPS = ["Walking", "Jogging", "Sitting", "Standing_GettingUp"]

# ---------------------------------------------------------------------------------------
# 1. KURTOGRAM-CELL INTENSITY -- proxy for "how impulsive/anomalous does this window look"
# ---------------------------------------------------------------------------------------
def kurtogram_intensity(filepath):
    """Extracts both kurtogram cells (Acc1 top-right, Gyro bottom-right) from a
    Mosaic_PatchCore_2x2 image and returns a single intensity score: standard deviation
    of pixel values away from a neutral gray baseline. A calm/normal window's kurtogram
    should be close to flat gray (low std); a genuine impact shows a sharp, high-contrast
    colored spike (high std)."""
    img = Image.open(filepath).convert("RGB")
    arr = np.array(img)
    if arr.shape[0] < KURTOGRAM_X1 or arr.shape[1] < KURTOGRAM_X1:
        w = arr.shape[1]
        kurt_region = arr[:, w // 2:]
    else:
        kurt_region = arr[:, KURTOGRAM_X0:KURTOGRAM_X1]
    return float(kurt_region.std())

def trial_id_from_filename(filepath):
    stem = os.path.splitext(os.path.basename(filepath))[0]
    return re.sub(r"_T[\d.]+-[\d.]+s$", "", stem)

def window_start_time(filepath):
    """Parses the window's start time (t0) from the '_T{t0}-{t1}s' filename suffix, used
    to establish TEMPORAL ORDER of windows within a trial. Returns None if no suffix found
    (single-window trial -- no ordering needed)."""
    stem = os.path.splitext(os.path.basename(filepath))[0]
    m = re.search(r"_T([\d.]+)-[\d.]+s$", stem)
    return float(m.group(1)) if m else None

# ---------------------------------------------------------------------------------------
# 2. REFINE EACH ACTIVITY'S test/anomaly FOLDER -- three-way classification
# ---------------------------------------------------------------------------------------
summary_rows = []
for activity_name in ACTIVITY_GROUPS:
    anomaly_dir = os.path.join(PATCHCORE_READY_ROOT, activity_name, "test", "anomaly")
    if not os.path.isdir(anomaly_dir):
        print(f"[WARN] No anomaly folder for {activity_name} -- skipping.")
        continue

    filepaths = glob.glob(os.path.join(anomaly_dir, "*.png"))
    if len(filepaths) == 0:
        print(f"[WARN] No anomaly images found for {activity_name} -- skipping.")
        continue

    print(f"\n{'='*70}\n{activity_name}\n{'='*70}")
    print(f"[INFO] Scoring kurtogram intensity for {len(filepaths)} images...")

    records = []
    for fp in filepaths:
        records.append({
            "filepath": fp,
            "trial_id": trial_id_from_filename(fp),
            "t0": window_start_time(fp),
            "kurt_intensity": kurtogram_intensity(fp),
        })
    df = pd.DataFrame(records)

    n_trials = df["trial_id"].nunique()
    print(f"[INFO] {len(df)} windows across {n_trials} fall trials "
          f"(~{len(df)/n_trials:.1f} windows/trial)")

    # Classify each trial's windows into: kept (impact), prefall_context (early, confident
    # normal), or excluded (ambiguous buffer + post-fall)
    label = pd.Series("excluded", index=df.index)  # default -- overwritten below
    for trial_id, group in df.groupby("trial_id"):
        group_sorted = group.sort_values("t0") if group["t0"].notna().all() else group
        # window "position" = its rank in time order within this trial (0 = earliest)
        positions = {idx: pos for pos, idx in enumerate(group_sorted.index)}

        top_k_idx = group.nlargest(min(TOP_K_WINDOWS_PER_TRIAL, len(group)), "kurt_intensity").index
        label.loc[top_k_idx] = "kept_anomaly"

        if group["t0"].notna().all() and len(top_k_idx) > 0:
            earliest_impact_pos = min(positions[idx] for idx in top_k_idx)
            for idx in group.index:
                if idx in top_k_idx:
                    continue
                pos = positions[idx]
                if pos < earliest_impact_pos - PREFALL_BUFFER_WINDOWS:
                    label.loc[idx] = "prefall_context"
                # else: stays "excluded" (ambiguous buffer, or after the impact)

    kept_df = df[label == "kept_anomaly"]
    prefall_df = df[label == "prefall_context"]
    excluded_df = df[label == "excluded"]

    print(f"[INFO] Kept as genuine anomaly     : {len(kept_df):>5} "
          f"({len(kept_df)/len(df)*100:.1f}%)")
    print(f"[INFO] Routed to prefall_context   : {len(prefall_df):>5} "
          f"({len(prefall_df)/len(df)*100:.1f}%)  -- confidently normal, evaluated separately")
    print(f"[INFO] Excluded (ambiguous/postfall): {len(excluded_df):>5} "
          f"({len(excluded_df)/len(df)*100:.1f}%)  -- not used anywhere")

    # Move prefall_context windows to their own new test category
    prefall_dir = os.path.join(PATCHCORE_READY_ROOT, activity_name, "test", "prefall_context")
    os.makedirs(prefall_dir, exist_ok=True)
    for fp in prefall_df["filepath"]:
        shutil.move(fp, os.path.join(prefall_dir, os.path.basename(fp)))

    # Move excluded windows to a separate folder (not deleted -- inspectable/reversible)
    excluded_dir = os.path.join(PATCHCORE_READY_ROOT, activity_name, "test", "anomaly_excluded_ambiguous_postfall")
    os.makedirs(excluded_dir, exist_ok=True)
    for fp in excluded_df["filepath"]:
        shutil.move(fp, os.path.join(excluded_dir, os.path.basename(fp)))

    summary_rows.append({
        "activity": activity_name,
        "original_anomaly_count": len(df),
        "n_trials": n_trials,
        "kept_as_anomaly": len(kept_df),
        "prefall_context": len(prefall_df),
        "excluded_ambiguous_postfall": len(excluded_df),
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 70 + "\nREFINEMENT SUMMARY\n" + "=" * 70)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(PATCHCORE_READY_ROOT, "anomaly_refinement_summary.csv"), index=False)

print(f"\n[DONE] test/anomaly now contains only high-confidence impact windows.")
print(f"[NEW]  test/prefall_context now contains confidently-normal pre-fall windows --")
print(f"       train_patchcore_models.py will report specificity on this set SEPARATELY")
print(f"       from your main Daily-Living test/good set.")
print(f"[NOTE] Nothing was deleted -- excluded windows are in")
print(f"       test/anomaly_excluded_ambiguous_postfall/ within each activity's folder.")
print(f"[NEXT STEP] Re-run train_patchcore_models.py.")

[INFO] Refining anomaly labels in: /kaggle/working/PatchCore_Ready

Walking
[INFO] Scoring kurtogram intensity for 3529 images...
[INFO] 3529 windows across 719 fall trials (~4.9 windows/trial)
[INFO] Kept as genuine anomaly     :  1438 (40.7%)
[INFO] Routed to prefall_context   :   650 (18.4%)  -- confidently normal, evaluated separately
[INFO] Excluded (ambiguous/postfall):  1441 (40.8%)  -- not used anywhere

Jogging
[INFO] Scoring kurtogram intensity for 596 images...
[INFO] 596 windows across 120 fall trials (~5.0 windows/trial)
[INFO] Kept as genuine anomaly     :   240 (40.3%)
[INFO] Routed to prefall_context   :    26 (4.4%)  -- confidently normal, evaluated separately
[INFO] Excluded (ambiguous/postfall):   330 (55.4%)  -- not used anywhere

Sitting
[INFO] Scoring kurtogram intensity for 3555 images...
[INFO] 3555 windows across 719 fall trials (~4.9 windows/trial)
[INFO] Kept as genuine anomaly     :  1438 (40.5%)
[INFO] Routed to prefall_context   :   119 (3.3%)  -- confiden

In [4]:
"""Train patchcore models · PY
===================================================================================================
PATCHCORE ANOMALY DETECTION -- 4 ACTIVITY-SPECIFIC MODELS (Walking, Jogging, Sitting, Standing_GettingUp)
FINAL VERSION -- includes two fixes derived from analyzing the first version's real results
===================================================================================================
Paste into ONE Kaggle notebook cell and run. Reads from the PatchCore_Ready/ folder built by
patchcore_dataset_organizer.py, AFTER refine_anomaly_windows.py has been run on it.

RUN ORDER (all three, in this order, in the same Kaggle session):
    1. patchcore_dataset_organizer.py   -- builds PatchCore_Ready/ from your raw mosaic dataset
    2. refine_anomaly_windows.py        -- fixes mislabeled pre-fall/post-fall windows (FIX 1 below)
    3. train_patchcore_models.py (this file) -- trains + evaluates, with balanced memory banks (FIX 2)

===================================================================================================
FIX 1 (in refine_anomaly_windows.py, run before this script) -- MISLABELED WINDOW PROBLEM
===================================================================================================
A 15-second fall trial with 50% overlap produces ~5 windows, but only 1-2 actually contain the
impact -- the rest are genuine pre-fall/post-fall NORMAL activity, mislabeled as "anomaly" simply
because they came from a Fall-labeled recording. This pollutes threshold calibration: the
threshold gets set low enough to accommodate these calm windows, which then ALSO flags genuinely
normal daily-living activity. Confirmed in a real run: Sitting's specificity was 0.00% (flagging
literally everything) before this fix, because at least one near-normal-looking "anomaly" window
dragged the threshold down catastrophically. refine_anomaly_windows.py uses the kurtogram cell's
own visual intensity (the same signal engineered to show a sharp spike at impact and flat color
elsewhere) to keep only the highest-confidence impact window(s) per trial, excluding the rest.

===================================================================================================
FIX 2 (in this file) -- BALANCED PER-SUB-ACTIVITY MEMORY BANKS
===================================================================================================
Even after Fix 1, Sitting's ROC-AUC barely moved (0.763 -> 0.759) while Walking/Jogging jumped to
~0.999 -- proving Sitting's problem was NOT the mislabeled-window issue, but something else: Sitting
pools FOUR different original activities (Sit_HalfHeight_Slow/Fast, Sit_LowHeight_Slow/Fast) into
one memory bank. Building the memory bank from this pooled mix gives no guarantee the coreset
algorithm picks a BALANCED sample across all four -- it could, by chance, end up mostly
representing one sub-type, leaving gaps in what the model considers "normal" for the others.

This script now builds ONE SEPARATE mini memory bank per original sub-activity (discovered from
the organizer's train/good/{sub_activity}/ subfolder structure), then concatenates them --
guaranteeing every sub-activity contributes its own fair share, rather than hoping the coreset
algorithm balances it naturally. For activities with only one source sub-activity (e.g.
Standing_GettingUp, sourced only from Collapse_Into_Chair), this is identical to the original
single-bank behavior -- no change for that case.

===================================================================================================
HOW PATCHCORE WORKS, IN PLAIN TERMS (unchanged from the first version, repeated for reference)
===================================================================================================
PatchCore has NO gradient training, NO backpropagation, NO loss function. Instead:
  1. A FROZEN, ImageNet-pretrained WideResNet-50 extracts local "patch" feature vectors from
     every train/good (normal) image -- one vector per small spatial region of the image.
  2. All those patch vectors form a "memory bank" of what normal patches look like (now built
     per sub-activity and concatenated -- see FIX 2 above).
  3. At test time, a new image's patches are compared against the memory bank -- for each patch,
     the distance to its NEAREST neighbor in the memory bank is computed. The image's final
     anomaly score = the WORST (maximum) of those patch distances.
  4. A threshold on this score decides fall vs. no-fall.

===================================================================================================
THE METHODOLOGICAL TRAP THIS SCRIPT DELIBERATELY AVOIDS -- please read this section
===================================================================================================
The naive way to guarantee "never miss a fall" is: pick the threshold as the LOWEST anomaly
score among your fall test images. That mathematically guarantees 100% recall -- but only on
THAT SAME test set (data leakage in threshold selection). This script splits test/good and
test/anomaly into a CALIBRATION half (threshold chosen here) and a separate FINAL HELD-OUT TEST
half (results reported from here, never seen during threshold selection) -- the reported numbers
are the honest, defensible ones for publication.
===================================================================================================
"""

import os
import re
import glob
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, roc_curve

# ---------------------------------------------------------------------------------------
# 0. CONFIG
# ---------------------------------------------------------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

PATCHCORE_IMG_SIZE = 224   # standard for backbone-based patch features (ImageNet-scale) --
                            # your mosaics are 456x456 natively, resized down here. Increase
                            # if you suspect fine kurtogram detail is being lost (costs more
                            # compute/memory -- more patches per image).
BACKBONE = "wide_resnet50_2"
FEATURE_LAYERS = ("layer2", "layer3")
CORESET_RATIO = 0.02        # keep 2% of the (pre-subsampled) patches as the memory bank
MAX_MEMORY_BANK_SIZE = 2000   # hard cap -- kept deliberately modest (see RAW_PATCH_SUBSAMPLE_CAP
                                # note below for the timing math that drove this number)
RAW_PATCH_SUBSAMPLE_CAP = 40000  # pre-subsample raw patches to this size before greedy coreset.
                                    # BENCHMARKED, not guessed: at 150,000 patches x 15,000 target,
                                    # greedy coreset measured ~2.6s/iteration on CPU -- extrapolated
                                    # to ~11 HOURS per activity just for this step, before even
                                    # accounting for the CPU-vs-GPU gap that caused the original
                                    # stall you hit. This cap (40,000 x 2,000 target = 80M distance
                                    # evaluations, ~28x cheaper than the original setting) is sized
                                    # to comfortably finish within a single GPU session. Uniform
                                    # random subsampling before the greedy step still preserves the
                                    # overall distribution of "normal" patches -- increase this later
                                    # if you have GPU budget to spare and want a larger memory bank.
BATCH_SIZE = 16
NUM_WORKERS = 2
CALIBRATION_FRACTION = 0.5   # 50% of test/good and test/anomaly used ONLY to pick the threshold

ACTIVITY_PRIORITY = ["Walking", "Sitting", "Standing_GettingUp", "Jogging"]  # per fall-type frequency

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device: {device}")

# ---------------------------------------------------------------------------------------
# 1. LOCATE THE ORGANIZED DATASET
# ---------------------------------------------------------------------------------------
def find_patchcore_ready_root():
    candidates = glob.glob("/kaggle/working/PatchCore_Ready") + \
                 glob.glob("/kaggle/input/**/PatchCore_Ready", recursive=True)
    if not candidates:
        raise FileNotFoundError(
            "Could not find 'PatchCore_Ready' -- run patchcore_dataset_organizer.py first, "
            "or attach its output as an input to this notebook."
        )
    return candidates[0]

PATCHCORE_READY_ROOT = find_patchcore_ready_root()
print(f"[INFO] Using organized dataset at: {PATCHCORE_READY_ROOT}")
OUTPUT_DIR = "/kaggle/working/patchcore_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------------------
# 2. DATASET / TRANSFORMS
# ---------------------------------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
eval_transform = transforms.Compose([
    transforms.Resize((PATCHCORE_IMG_SIZE, PATCHCORE_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class ImageListDataset(Dataset):
    def __init__(self, filepaths, transform):
        self.filepaths = list(filepaths)
        self.transform = transform
    def __len__(self):
        return len(self.filepaths)
    def __getitem__(self, i):
        img = Image.open(self.filepaths[i]).convert("RGB")
        return self.transform(img), self.filepaths[i]

def trial_id_from_filename(filepath):
    """Strips the trailing '_T{t0}-{t1}s' window suffix (if present) so windows from the
    SAME original recording group together for trial-level analysis."""
    stem = os.path.splitext(os.path.basename(filepath))[0]
    return re.sub(r"_T[\d.]+-[\d.]+s$", "", stem)

# ---------------------------------------------------------------------------------------
# 3. PATCH FEATURE EXTRACTOR -- frozen WideResNet-50, layer2+layer3, local avg-pool, concat
# ---------------------------------------------------------------------------------------
class PatchFeatureExtractor(nn.Module):
    def __init__(self, backbone_name=BACKBONE, layers=FEATURE_LAYERS):
        super().__init__()
        backbone = self._load_pretrained(backbone_name)
        self.layers = layers
        self._features = {}
        for name in layers:
            getattr(backbone, name).register_forward_hook(self._make_hook(name))
        self.backbone = backbone
        self.backbone.eval()
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.pool = nn.AvgPool2d(kernel_size=3, stride=1, padding=1)

    def _load_pretrained(self, name, max_retries=3):
        """Same robust retry-with-cache-clear pattern used in the ADL classifier scripts."""
        import time
        cache_dir = os.path.expanduser("~/.cache/torch/hub/checkpoints")
        weight_enums = {
            "wide_resnet50_2": models.Wide_ResNet50_2_Weights.IMAGENET1K_V1,
            "resnet18": models.ResNet18_Weights.IMAGENET1K_V1,
            "resnet50": models.ResNet50_Weights.IMAGENET1K_V1,
        }
        if name not in weight_enums:
            raise ValueError(f"Unsupported backbone '{name}' -- add it to weight_enums first.")
        for attempt in range(1, max_retries + 1):
            try:
                print(f"[INFO] Loading {name} pretrained weights (attempt {attempt}/{max_retries})...")
                m = getattr(models, name)(weights=weight_enums[name])
                print("[INFO] Pretrained weights loaded successfully.")
                return m
            except (RuntimeError, OSError) as e:
                print(f"[WARN] Weight download/verification failed: {type(e).__name__}: {e}")
                if os.path.isdir(cache_dir):
                    for fname in os.listdir(cache_dir):
                        if name.replace("_", "") in fname.replace("_", ""):
                            try:
                                os.remove(os.path.join(cache_dir, fname))
                            except OSError:
                                pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        print(f"[WARN] All retries failed for {name} -- falling back to random init (NOT recommended for PatchCore).")
        return getattr(models, name)(weights=None)

    def _make_hook(self, name):
        def hook(module, inp, out):
            self._features[name] = out
        return hook

    @torch.no_grad()
    def forward(self, x):
        self._features = {}
        _ = self.backbone(x)
        feats = [self._features[n] for n in self.layers]
        feats = [self.pool(f) for f in feats]
        target_size = feats[0].shape[-2:]
        feats = [F.interpolate(f, size=target_size, mode="bilinear", align_corners=False)
                 if f.shape[-2:] != target_size else f for f in feats]
        cat = torch.cat(feats, dim=1)          # (B, C, H, W)
        B, C, H, W = cat.shape
        patches = cat.permute(0, 2, 3, 1).reshape(B, H * W, C)   # (B, num_patches, C)
        return patches

feature_extractor = PatchFeatureExtractor().to(device)

# ---------------------------------------------------------------------------------------
# 4. GREEDY CORESET SELECTION -- k-center farthest-point sampling
# ---------------------------------------------------------------------------------------
@torch.no_grad()
def greedy_coreset(features, target_size, device, chunk_size=4096):
    """features: (N, C) tensor. Returns (target_size, C) subset -- a genuine subset of the
    input, never fabricated points -- selected to maximally cover the feature space."""
    features = features.to(device)
    n = features.shape[0]
    target_size = min(target_size, n)
    start_idx = 0
    selected_idx = [start_idx]
    min_dists = torch.cdist(features, features[start_idx:start_idx + 1]).squeeze(1)
    for _ in range(target_size - 1):
        next_idx = torch.argmax(min_dists).item()
        selected_idx.append(next_idx)
        new_dists = torch.cdist(features, features[next_idx:next_idx + 1]).squeeze(1)
        min_dists = torch.minimum(min_dists, new_dists)
        min_dists[next_idx] = -1
    return features[selected_idx].cpu(), selected_idx

# ---------------------------------------------------------------------------------------
# 5. BUILD MEMORY BANK FROM train/good IMAGES
# ---------------------------------------------------------------------------------------
@torch.no_grad()
def extract_all_patches(filepaths, desc=""):
    ds = ImageListDataset(filepaths, eval_transform)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    all_patches = []
    for imgs, _ in loader:
        imgs = imgs.to(device)
        patches = feature_extractor(imgs)              # (B, num_patches, C)
        B, P, C = patches.shape
        all_patches.append(patches.reshape(B * P, C).cpu())
    return torch.cat(all_patches, dim=0)

def build_memory_bank(train_good_by_subactivity, activity_name):
    """train_good_by_subactivity: dict of {sub_activity_name: [filepaths]}.

    Builds ONE SEPARATE mini memory bank per sub-activity (e.g. for 'Sitting':
    Sit_HalfHeight_Slow, Sit_HalfHeight_Fast, Sit_LowHeight_Slow, Sit_LowHeight_Fast
    each get their own coreset selection), then concatenates them into one final
    memory bank for the activity group.

    WHY THIS MATTERS (this is the fix for the low-specificity problem diagnosed on
    Sitting/Standing_GettingUp): if all sub-activities were pooled together BEFORE
    coreset selection, the greedy farthest-point algorithm has no guarantee of
    picking a balanced number of representative points from each sub-activity --
    it could, by chance, end up mostly representing whichever sub-activity has more
    or more-diverse raw patches, leaving the others under-represented as "normal".
    Building separate mini-banks and concatenating GUARANTEES every sub-activity
    contributes its own fair share to what counts as "normal", regardless of how
    the original pool was distributed.

    For an activity with only ONE sub-activity (e.g. Standing_GettingUp, which only
    has Collapse_Into_Chair as its source), this reduces to exactly the same
    single-bank behavior as before -- no change for that case.
    """
    sub_banks = []
    for sub_activity, filepaths in train_good_by_subactivity.items():
        print(f"\n  [Sub-activity: {sub_activity}] Extracting patch features from "
              f"{len(filepaths)} images...")
        all_patches = extract_all_patches(filepaths, desc=f"{activity_name}/{sub_activity}")
        print(f"  [Sub-activity: {sub_activity}] Total raw patches: {all_patches.shape[0]:,} "
              f"(dim={all_patches.shape[1]})")

        # Pre-subsample BEFORE greedy coreset if there are too many raw patches -- this is
        # what the original PatchCore paper does for tractability, not a shortcut. Uniform
        # random subsampling preserves the overall distribution of "normal" patches; the
        # greedy step afterward still picks the most diverse/representative subset of
        # WHATEVER pool it's given.
        if all_patches.shape[0] > RAW_PATCH_SUBSAMPLE_CAP:
            rng = np.random.default_rng(SEED)
            keep_idx = rng.choice(all_patches.shape[0], size=RAW_PATCH_SUBSAMPLE_CAP, replace=False)
            all_patches = all_patches[keep_idx]
            print(f"  [Sub-activity: {sub_activity}] Pre-subsampled to {all_patches.shape[0]:,} "
                  f"before coreset (cap={RAW_PATCH_SUBSAMPLE_CAP:,}).")

        # Each sub-activity gets its OWN fixed target size (not divided by number of
        # sub-activities) -- this is the guarantee of balanced representation. A
        # 'Sitting' group with 4 sub-activities ends up with a LARGER total memory
        # bank (4 x MAX_MEMORY_BANK_SIZE) than a single-sub-activity group like
        # Standing_GettingUp (1 x MAX_MEMORY_BANK_SIZE) -- proportional to how many
        # genuinely distinct normal patterns that activity group actually contains.
        target_size = min(int(all_patches.shape[0] * CORESET_RATIO), MAX_MEMORY_BANK_SIZE)
        print(f"  [Sub-activity: {sub_activity}] Coreset target: {target_size:,}")
        sub_bank, _ = greedy_coreset(all_patches, target_size, device)
        print(f"  [Sub-activity: {sub_activity}] Mini memory bank size: {sub_bank.shape[0]:,}")
        sub_banks.append(sub_bank)

    memory_bank = torch.cat(sub_banks, dim=0)
    print(f"\n[INFO] Combined memory bank for '{activity_name}': {memory_bank.shape[0]:,} total "
          f"points across {len(sub_banks)} sub-activity/activities")
    return memory_bank

# ---------------------------------------------------------------------------------------
# 6. SCORE IMAGES AGAINST A MEMORY BANK
# ---------------------------------------------------------------------------------------
@torch.no_grad()
def score_images(filepaths, memory_bank, chunk_size=2048):
    """Returns one anomaly score per image: max over patches of (min distance to memory bank)."""
    memory_bank = memory_bank.to(device)
    ds = ImageListDataset(filepaths, eval_transform)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    scores = []
    ordered_filepaths = []
    for imgs, fps in loader:
        imgs = imgs.to(device)
        patches = feature_extractor(imgs)          # (B, num_patches, C)
        B, P, C = patches.shape
        for b in range(B):
            p = patches[b]                          # (num_patches, C)
            dists = torch.cdist(p, memory_bank)      # (num_patches, memory_bank_size)
            min_dists = dists.min(dim=1).values      # nearest-neighbor distance per patch
            image_score = min_dists.max().item()     # worst patch = image anomaly score
            scores.append(image_score)
        ordered_filepaths.extend(fps)
    return np.array(scores), ordered_filepaths

# ---------------------------------------------------------------------------------------
# 7. CALIBRATION / HELD-OUT SPLIT -- avoids the threshold data-snooping trap (see docstring)
# ---------------------------------------------------------------------------------------
def calibration_split(filepaths, calib_frac=CALIBRATION_FRACTION, seed=SEED):
    idx = np.arange(len(filepaths))
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    n_calib = int(len(idx) * calib_frac)
    calib = [filepaths[i] for i in idx[:n_calib]]
    final_test = [filepaths[i] for i in idx[n_calib:]]
    return calib, final_test

# ---------------------------------------------------------------------------------------
# 8. RUN ONE ACTIVITY END-TO-END
# ---------------------------------------------------------------------------------------
def run_activity(activity_name):
    print(f"\n{'='*70}\n{activity_name}\n{'='*70}")
    act_root = os.path.join(PATCHCORE_READY_ROOT, activity_name)

    # train/good is now organized as train/good/{sub_activity}/*.png (see the updated
    # organizer script) -- discover whichever sub-activity subfolders exist for this
    # activity group, so build_memory_bank can build one balanced mini-bank per
    # sub-activity rather than one pooled bank that risks under-representing some of them.
    train_good_root = os.path.join(act_root, "train", "good")
    sub_activity_dirs = sorted([
        d for d in os.listdir(train_good_root)
        if os.path.isdir(os.path.join(train_good_root, d))
    ]) if os.path.isdir(train_good_root) else []
    train_good_by_subactivity = {
        sub: glob.glob(os.path.join(train_good_root, sub, "*.png"))
        for sub in sub_activity_dirs
    }
    total_train_good = sum(len(v) for v in train_good_by_subactivity.values())

    test_good = glob.glob(os.path.join(act_root, "test", "good", "*.png"))
    test_anomaly = glob.glob(os.path.join(act_root, "test", "anomaly", "*.png"))
    print(f"[INFO] train/good={total_train_good} across {len(sub_activity_dirs)} sub-activities "
          f"{sub_activity_dirs}")
    print(f"[INFO] test/good={len(test_good)}  test/anomaly={len(test_anomaly)}")
    if total_train_good == 0 or len(test_anomaly) == 0:
        print(f"[WARN] Missing data for {activity_name} -- skipping.")
        return None

    # Step 1: build the memory bank from train/good only (one-class -- never sees anomalies)
    memory_bank = build_memory_bank(train_good_by_subactivity, activity_name)
    torch.save(memory_bank, os.path.join(OUTPUT_DIR, f"memory_bank_{activity_name}.pt"))

    # Step 2: split test/good and test/anomaly into calibration vs. final held-out test
    good_calib, good_final = calibration_split(test_good)
    anom_calib, anom_final = calibration_split(test_anomaly)
    print(f"[INFO] Calibration -> good:{len(good_calib)} anomaly:{len(anom_calib)}")
    print(f"[INFO] Final held-out test -> good:{len(good_final)} anomaly:{len(anom_final)}")

    # Step 3: score everything
    good_calib_scores, _ = score_images(good_calib, memory_bank)
    anom_calib_scores, _ = score_images(anom_calib, memory_bank)
    good_final_scores, good_final_fps = score_images(good_final, memory_bank)
    anom_final_scores, anom_final_fps = score_images(anom_final, memory_bank)

    # Step 4: pick threshold on CALIBRATION set only -- guarantees 100% recall on calibration,
    # but the threshold is chosen WITHOUT looking at the final held-out test set at all.
    threshold = float(anom_calib_scores.min()) if len(anom_calib_scores) > 0 else float(good_calib_scores.max())
    calib_recall = (anom_calib_scores >= threshold).mean()
    print(f"[INFO] Threshold (from calibration set): {threshold:.4f}  "
          f"(calibration recall by construction: {calib_recall*100:.1f}%)")

    # Step 5: report the HONEST numbers on the held-out final test set
    final_recall = (anom_final_scores >= threshold).mean()      # sensitivity -- never-miss-a-fall metric
    final_fpr = (good_final_scores >= threshold).mean()          # false alarm rate on final test
    final_specificity = 1 - final_fpr
    print(f"\n[RESULT] FINAL HELD-OUT TEST (threshold never saw these images):")
    print(f"  Recall / Sensitivity (falls correctly caught) : {final_recall*100:.2f}%")
    print(f"  Specificity (normal activity correctly cleared): {final_specificity*100:.2f}%")
    print(f"  False Positive Rate on normal activity          : {final_fpr*100:.2f}%")

    # Trial-level recall -- did every ORIGINAL fall recording have at least one window flagged?
    trial_scores = {}
    for fp, score in zip(anom_final_fps, anom_final_scores):
        tid = trial_id_from_filename(fp)
        trial_scores.setdefault(tid, []).append(score)
    trial_caught = [max(scores) >= threshold for scores in trial_scores.values()]
    trial_recall = np.mean(trial_caught) if trial_caught else float("nan")
    print(f"  Trial-level recall (>=1 window flagged per fall event): {trial_recall*100:.2f}% "
          f"({sum(trial_caught)}/{len(trial_caught)} trials)")

    # NEW: prefall_context evaluation -- confidently-normal windows from the moments right
    # before a fall (built by refine_anomaly_windows.py). Reported SEPARATELY from the main
    # test/good specificity, since this is a more specific, more realistic false-positive
    # scenario: "does the model stay calm in the moments right before a fall happens",
    # rather than just "does it false-alarm on random daily activity in general."
    prefall_dir = os.path.join(act_root, "test", "prefall_context")
    prefall_specificity = None
    if os.path.isdir(prefall_dir):
        prefall_files = glob.glob(os.path.join(prefall_dir, "*.png"))
        if len(prefall_files) > 0:
            prefall_scores, _ = score_images(prefall_files, memory_bank)
            prefall_fpr = (prefall_scores >= threshold).mean()
            prefall_specificity = 1 - prefall_fpr
            print(f"  Specificity on PRE-FALL CONTEXT windows ({len(prefall_files)} windows, "
                  f"confidently normal activity right before a fall): {prefall_specificity*100:.2f}%")
        else:
            print(f"  [INFO] No prefall_context windows found for {activity_name} "
                  f"(run refine_anomaly_windows.py to generate this category).")
    else:
        print(f"  [INFO] No prefall_context folder found -- run refine_anomaly_windows.py "
              f"(updated version) to enable this evaluation.")

    # ROC-AUC -- threshold-independent separability measure, combining calib+final for a
    # larger, more stable estimate (this metric doesn't depend on the specific threshold chosen)
    all_scores = np.concatenate([good_calib_scores, anom_calib_scores, good_final_scores, anom_final_scores])
    all_labels = np.concatenate([
        np.zeros(len(good_calib_scores)), np.ones(len(anom_calib_scores)),
        np.zeros(len(good_final_scores)), np.ones(len(anom_final_scores)),
    ])
    auc = roc_auc_score(all_labels, all_scores)
    print(f"  ROC-AUC (threshold-independent separability)   : {auc:.4f}")

    # NEW: normalized 0-1 anomaly score, for human-readable reporting/visualization.
    # Min-max scaled using the CALIBRATION set's score range (never the final-test set,
    # to avoid any snooping) -- 0.0 = as normal-looking as the calmest calibration example,
    # 1.0 = as anomalous-looking as the most extreme calibration fall example. This is
    # PURELY for display/interpretability -- the actual fall/no-fall decision still uses
    # the raw 'threshold' value above, not this normalized version.
    calib_score_min = float(min(good_calib_scores.min(), anom_calib_scores.min()))
    calib_score_max = float(max(good_calib_scores.max(), anom_calib_scores.max()))
    def normalize_score(raw_score):
        if calib_score_max <= calib_score_min:
            return 0.5
        return float(np.clip((raw_score - calib_score_min) / (calib_score_max - calib_score_min), 0, 1))
    normalized_threshold = normalize_score(threshold)
    print(f"  [INFO] Normalized threshold (0-1 scale, for display purposes): {normalized_threshold:.3f}")

    # Plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(good_final_scores, bins=30, alpha=0.6, label="Normal (held-out)", color="steelblue")
    axes[0].hist(anom_final_scores, bins=30, alpha=0.6, label="Fall (held-out)", color="crimson")
    axes[0].axvline(threshold, color="black", linestyle="--", label=f"Threshold={threshold:.3f}")
    axes[0].set_xlabel("Anomaly score"); axes[0].set_ylabel("Count")
    axes[0].set_title(f"{activity_name}: Score Distribution (final held-out test)")
    axes[0].legend()

    fpr_curve, tpr_curve, _ = roc_curve(all_labels, all_scores)
    axes[1].plot(fpr_curve, tpr_curve, label=f"AUC={auc:.4f}")
    axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
    axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title(f"{activity_name}: ROC Curve")
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"patchcore_{activity_name}_analysis.png"), dpi=150)
    plt.show()

    return {
        "activity": activity_name,
        "train_good_n": total_train_good,
        "memory_bank_size": memory_bank.shape[0],
        "threshold": threshold,
        "normalized_threshold": normalized_threshold,
        "calib_score_min": calib_score_min,
        "calib_score_max": calib_score_max,
        "final_recall": final_recall,
        "final_specificity": final_specificity,
        "final_fpr": final_fpr,
        "prefall_context_specificity": prefall_specificity,
        "trial_recall": trial_recall,
        "n_trials": len(trial_caught),
        "roc_auc": auc,
    }

# ---------------------------------------------------------------------------------------
# 9. RUN ALL 4 ACTIVITIES, IN FALL-TYPE-FREQUENCY PRIORITY ORDER
# ---------------------------------------------------------------------------------------
results = []
for activity_name in ACTIVITY_PRIORITY:
    r = run_activity(activity_name)
    if r is not None:
        results.append(r)

results_df = pd.DataFrame(results)
print("\n" + "=" * 90 + "\nFINAL SUMMARY -- ALL ACTIVITY-SPECIFIC PATCHCORE MODELS\n" + "=" * 90)
print(results_df.to_string(index=False))
results_df.to_csv(os.path.join(OUTPUT_DIR, "patchcore_summary.csv"), index=False)

print(f"\n[DONE] All PatchCore artifacts saved under {OUTPUT_DIR}:")
print("  - memory_bank_{Activity}.pt      (the trained one-class model per activity)")
print("  - patchcore_{Activity}_analysis.png  (score histogram + ROC curve)")
print("  - patchcore_summary.csv          (all 4 activities' final metrics)")
print("\n[REMINDER] 'final_recall' and 'trial_recall' above were measured on a held-out test")
print("           split the threshold never saw -- these are the numbers to trust and report.")

# ---------------------------------------------------------------------------------------
# 10. INTEGRATION EXAMPLE -- how this connects to your ADL classifier (train_final_v4.py)
# ---------------------------------------------------------------------------------------
"""
Conceptual pipeline for a new incoming 5-second window:

    1. Generate BOTH mosaics for this window (Mosaic_ADL_2x2 AND Mosaic_PatchCore_2x2)
       from the same raw sensor slice, exactly as your generation pipeline already does.

    2. Run the ADL classifier (train_final_v4.py's model) on the Mosaic_ADL_2x2 image.
       predicted_activity = model.predict(mosaic_adl_image)   # e.g. "Walking"

    3. Map predicted_activity to its PatchCore activity group:
       ACTIVITY_TO_PATCHCORE_GROUP = {
           "Walking_Slow": "Walking", "Walking_Fast": "Walking",
           "Jogging": "Jogging",
           "Sitting": "Sitting",
           "Collapse_Into_Chair": "Standing_GettingUp",
           # other classes (Stairs, Standing_KneeBend, etc.) -> no PatchCore model yet;
           # decide a fallback policy (e.g. route to nearest available group, or flag
           # as "unmonitored activity" until you build models for them too)
       }
       group = ACTIVITY_TO_PATCHCORE_GROUP.get(predicted_activity)

    4. Load that group's memory bank (already saved to disk above) and score the SAME
       window's Mosaic_PatchCore_2x2 image against it:
       memory_bank = torch.load(f"memory_bank_{group}.pt")
       score, _ = score_images([mosaic_patchcore_path], memory_bank)

    5. Compare against that group's calibrated threshold (from patchcore_summary.csv):
       is_fall = score[0] >= threshold_for(group)
"""

[INFO] Device: cuda
[INFO] Using organized dataset at: /kaggle/working/PatchCore_Ready
[INFO] Loading wide_resnet50_2 pretrained weights (attempt 1/3)...
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth


100%|██████████| 132M/132M [00:00<00:00, 193MB/s]


[INFO] Pretrained weights loaded successfully.

Walking
[INFO] train/good=2387 across 2 sub-activities ['Walking_Fast', 'Walking_Slow']
[INFO] test/good=597  test/anomaly=1438

  [Sub-activity: Walking_Fast] Extracting patch features from 1204 images...
  [Sub-activity: Walking_Fast] Total raw patches: 943,936 (dim=1536)
  [Sub-activity: Walking_Fast] Pre-subsampled to 40,000 before coreset (cap=40,000).
  [Sub-activity: Walking_Fast] Coreset target: 800
  [Sub-activity: Walking_Fast] Mini memory bank size: 800

  [Sub-activity: Walking_Slow] Extracting patch features from 1183 images...
  [Sub-activity: Walking_Slow] Total raw patches: 927,472 (dim=1536)
  [Sub-activity: Walking_Slow] Pre-subsampled to 40,000 before coreset (cap=40,000).
  [Sub-activity: Walking_Slow] Coreset target: 800
  [Sub-activity: Walking_Slow] Mini memory bank size: 800

[INFO] Combined memory bank for 'Walking': 1,600 total points across 2 sub-activity/activities
[INFO] Calibration -> good:298 anomaly:719
[IN

'\nConceptual pipeline for a new incoming 5-second window:\n\n    1. Generate BOTH mosaics for this window (Mosaic_ADL_2x2 AND Mosaic_PatchCore_2x2)\n       from the same raw sensor slice, exactly as your generation pipeline already does.\n\n    2. Run the ADL classifier (train_final_v4.py\'s model) on the Mosaic_ADL_2x2 image.\n       predicted_activity = model.predict(mosaic_adl_image)   # e.g. "Walking"\n\n    3. Map predicted_activity to its PatchCore activity group:\n       ACTIVITY_TO_PATCHCORE_GROUP = {\n           "Walking_Slow": "Walking", "Walking_Fast": "Walking",\n           "Jogging": "Jogging",\n           "Sitting": "Sitting",\n           "Collapse_Into_Chair": "Standing_GettingUp",\n           # other classes (Stairs, Standing_KneeBend, etc.) -> no PatchCore model yet;\n           # decide a fallback policy (e.g. route to nearest available group, or flag\n           # as "unmonitored activity" until you build models for them too)\n       }\n       group = ACTIVITY_TO_PA

In [5]:
"""Patchcore heatmap visualization · PY
===================================================================================================
PATCHCORE HEATMAP VISUALIZATION -- FINAL VERSION
===================================================================================================
Run this AFTER train_patchcore_models.py (needs saved memory_bank_{Activity}.pt files AND
patchcore_summary.csv, so the threshold/normalization here EXACTLY matches your real trained
model's decision boundary, not a recomputed approximation).

WHAT CHANGED FROM THE PREVIOUS VERSION -- and why each change matters:

FIX 1 -- CONSISTENT COLOR SCALE (this explains the "normal image shows abnormality" confusion).
  The previous version let matplotlib auto-scale each image's heatmap colors to that image's
  OWN min/max. This means even a perfectly calm, genuinely normal image always showed a full
  blue-to-red gradient, because *something* has to be the "relatively highest" value in any
  image, regardless of how small the real numbers are. This version uses ONE shared, fixed
  color scale (based on the real calibration score range from patchcore_summary.csv) across
  EVERY image shown -- now a truly calm image shows mostly uniform cool colors, and only a
  genuine fall shows real, comparable red.

FIX 2 -- PEAK LOCATION + QUADRANT IDENTIFICATION (answers "which specific spot indicates fall").
  Every heatmap now has its single hottest point marked with a white circle, and labeled with
  which of the 4 mosaic quadrants it falls in: Acc1-Scalogram, Acc1-Kurtogram, Gyro-Scalogram,
  or Gyro-Kurtogram. If a FALL prediction's hot spot is NOT on a Kurtogram quadrant, that's
  genuinely informative -- it means the model flagged this image for a different reason than
  the impact signature the kurtogram was specifically designed to detect, worth investigating
  further rather than assuming it's simply correct.

FIX 3 -- GROUND TRUTH vs. PREDICTION, shown separately and explicitly.
  Every image now shows BOTH: "Ground Truth" (which folder it actually came from: good,
  prefall_context, or anomaly) AND "Predicted" (fall/normal, from comparing its score against
  the REAL trained threshold loaded from patchcore_summary.csv). This is what lets you see
  true/false positives and negatives directly, not just "here's a heatmap."

FIX 4 -- NO MORE OVERLAPPING TEXT. Figure sizes and title placement were reworked with explicit
  spacing (tight_layout with reserved title space, shorter per-image titles, larger figures).

FIX 5 -- MORE SAMPLES, configurable (N_EXAMPLES_PER_CLASS increased and easy to raise further).

NEW -- 15-IMAGE INDIVIDUAL TEST SUITE (the "different folder, 5+5+5" request):
  For each activity, 5 images are randomly sampled from EACH of: test/good (ground truth
  normal), test/prefall_context (ground truth normal, pre-fall specifically), and test/anomaly
  (ground truth fall) -- 15 total. EVERY ONE of these 15 images gets its OWN individual, fully
  labeled output file (original + heatmap + ground truth + prediction + score + peak quadrant),
  saved into a folder structure organized by activity and category, PLUS one combined CSV
  summarizing all 15 images' results for a quick overview without opening every file.
===================================================================================================
"""

import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

# ---------------------------------------------------------------------------------------
# 0. CONFIG
# ---------------------------------------------------------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

PATCHCORE_IMG_SIZE = 224
BACKBONE = "wide_resnet50_2"
FEATURE_LAYERS = ("layer2", "layer3")
N_EXAMPLES_PER_CLASS = 8          # overview grid: increased from 4 -> 8, easy to raise further
N_INDIVIDUAL_TEST_PER_CATEGORY = 5  # the "5+5+5=15" individual deep-dive test
ACTIVITY_PRIORITY = ["Walking", "Sitting", "Standing_GettingUp", "Jogging"]

# Mosaic quadrant geometry (verified against the actual generation pipeline: 456x456
# canvas, 224px cells, 8px gutter -- scaled down to the 224x224 processed image size)
CELL_SIZE, GUTTER_PX, FULL_CANVAS = 224, 8, 456
SPLIT_PX = round(PATCHCORE_IMG_SIZE * (CELL_SIZE + GUTTER_PX) / FULL_CANVAS)  # = 114

def quadrant_name(row, col):
    vertical = "Acc1" if row < SPLIT_PX else "Gyro"
    horizontal = "Scalogram" if col < SPLIT_PX else "Kurtogram"
    return f"{vertical}-{horizontal}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device: {device}")

def find_patchcore_ready_root():
    candidates = glob.glob("/kaggle/working/PatchCore_Ready") + \
                 glob.glob("/kaggle/input/**/PatchCore_Ready", recursive=True)
    if not candidates:
        raise FileNotFoundError("Could not find 'PatchCore_Ready' -- run the organizer + refiner first.")
    return candidates[0]

def find_memory_bank(activity_name):
    candidates = (
        glob.glob(f"/kaggle/working/**/memory_bank_{activity_name}.pt", recursive=True) +
        glob.glob(f"/kaggle/input/**/memory_bank_{activity_name}.pt", recursive=True)
    )
    if not candidates:
        raise FileNotFoundError(f"Could not find memory_bank_{activity_name}.pt -- run train_patchcore_models.py first.")
    return candidates[0]

def find_patchcore_summary():
    candidates = glob.glob("/kaggle/working/**/patchcore_summary.csv", recursive=True) + \
                 glob.glob("/kaggle/input/**/patchcore_summary.csv", recursive=True)
    if not candidates:
        raise FileNotFoundError(
            "Could not find patchcore_summary.csv -- run train_patchcore_models.py first. "
            "This file's threshold/calibration range is required so predictions here exactly "
            "match your real trained model, rather than an approximation."
        )
    return pd.read_csv(candidates[0]).set_index("activity")

PATCHCORE_READY_ROOT = find_patchcore_ready_root()
SUMMARY_DF = find_patchcore_summary()
OUTPUT_DIR = "/kaggle/working/heatmap_visualizations"
INDIVIDUAL_DIR = os.path.join(OUTPUT_DIR, "individual_test_15")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(INDIVIDUAL_DIR, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
eval_transform = transforms.Compose([
    transforms.Resize((PATCHCORE_IMG_SIZE, PATCHCORE_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ---------------------------------------------------------------------------------------
# 1. FEATURE EXTRACTOR (identical to train_patchcore_models.py)
# ---------------------------------------------------------------------------------------
class PatchFeatureExtractor(nn.Module):
    def __init__(self, backbone_name=BACKBONE, layers=FEATURE_LAYERS):
        super().__init__()
        backbone = self._load_pretrained(backbone_name)
        self.layers = layers
        self._features = {}
        for name in layers:
            getattr(backbone, name).register_forward_hook(self._make_hook(name))
        self.backbone = backbone
        self.backbone.eval()
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.pool = nn.AvgPool2d(kernel_size=3, stride=1, padding=1)

    def _load_pretrained(self, name, max_retries=3):
        import time
        cache_dir = os.path.expanduser("~/.cache/torch/hub/checkpoints")
        weight_enums = {"wide_resnet50_2": models.Wide_ResNet50_2_Weights.IMAGENET1K_V1}
        for attempt in range(1, max_retries + 1):
            try:
                print(f"[INFO] Loading {name} pretrained weights (attempt {attempt}/{max_retries})...")
                m = getattr(models, name)(weights=weight_enums[name])
                print("[INFO] Pretrained weights loaded successfully.")
                return m
            except (RuntimeError, OSError) as e:
                print(f"[WARN] Weight download/verification failed: {type(e).__name__}: {e}")
                if os.path.isdir(cache_dir):
                    for fname in os.listdir(cache_dir):
                        if name.replace("_", "") in fname.replace("_", ""):
                            try:
                                os.remove(os.path.join(cache_dir, fname))
                            except OSError:
                                pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        print(f"[WARN] All retries failed -- falling back to random init.")
        return getattr(models, name)(weights=None)

    def _make_hook(self, name):
        def hook(module, inp, out):
            self._features[name] = out
        return hook

    @torch.no_grad()
    def forward(self, x):
        self._features = {}
        _ = self.backbone(x)
        feats = [self._features[n] for n in self.layers]
        feats = [self.pool(f) for f in feats]
        target_size = feats[0].shape[-2:]
        feats = [F.interpolate(f, size=target_size, mode="bilinear", align_corners=False)
                 if f.shape[-2:] != target_size else f for f in feats]
        cat = torch.cat(feats, dim=1)
        B, C, H, W = cat.shape
        patches = cat.permute(0, 2, 3, 1).reshape(B, H * W, C)
        return patches, (H, W)

feature_extractor = PatchFeatureExtractor().to(device)

# ---------------------------------------------------------------------------------------
# 2. PER-PATCH ANOMALY MAP + PEAK LOCALIZATION
# ---------------------------------------------------------------------------------------
@torch.no_grad()
def compute_anomaly_heatmap(filepath, memory_bank):
    img = Image.open(filepath).convert("RGB")
    img_resized = img.resize((PATCHCORE_IMG_SIZE, PATCHCORE_IMG_SIZE))
    input_tensor = eval_transform(img).unsqueeze(0).to(device)

    patches, (H, W) = feature_extractor(input_tensor)
    patches = patches.squeeze(0)

    memory_bank_dev = memory_bank.to(device)
    dists = torch.cdist(patches, memory_bank_dev)
    min_dists = dists.min(dim=1).values

    image_score = min_dists.max().item()

    heatmap_grid = min_dists.reshape(1, 1, H, W)
    heatmap_upscaled = F.interpolate(heatmap_grid, size=(PATCHCORE_IMG_SIZE, PATCHCORE_IMG_SIZE),
                                      mode="bilinear", align_corners=False)
    heatmap_upscaled = heatmap_upscaled.squeeze().cpu().numpy()

    peak_row, peak_col = np.unravel_index(np.argmax(heatmap_upscaled), heatmap_upscaled.shape)
    peak_quadrant = quadrant_name(peak_row, peak_col)

    return img_resized, heatmap_upscaled, image_score, (peak_row, peak_col), peak_quadrant

def normalize_score(raw_score, calib_min, calib_max):
    if calib_max <= calib_min:
        return 0.5
    return float(np.clip((raw_score - calib_min) / (calib_max - calib_min), 0, 1))

def plot_one_result(ax_img, ax_heat, img_resized, heatmap, score, norm_score, peak_rc,
                     peak_quadrant, ground_truth, predicted, threshold, vmin, vmax):
    """Draws one image+heatmap pair with full labeling. vmin/vmax are FIXED and SHARED
    across a whole comparison set -- this is FIX 1, the consistent color scale."""
    ax_img.imshow(img_resized)
    gt_color = "green" if ground_truth == "Normal" else "crimson"
    ax_img.set_title(f"Ground Truth: {ground_truth}", color=gt_color, fontsize=10, fontweight="bold")
    ax_img.axis("off")

    ax_heat.imshow(img_resized)
    im = ax_heat.imshow(heatmap, cmap="jet", alpha=0.45, vmin=vmin, vmax=vmax)
    ax_heat.scatter([peak_rc[1]], [peak_rc[0]], s=120, facecolors="none",
                     edgecolors="white", linewidths=2)
    pred_color = "crimson" if predicted == "FALL" else "green"
    correct = (ground_truth == "Fall" and predicted == "FALL") or (ground_truth == "Normal" and predicted == "NORMAL")
    match_str = "CORRECT" if correct else "MISMATCH"
    ax_heat.set_title(
        f"Predicted: {predicted} ({match_str})\n"
        f"raw={score:.3f} (threshold={threshold:.3f}) | normalized={norm_score:.3f}\n"
        f"Peak region: {peak_quadrant}",
        color=pred_color, fontsize=9
    )
    ax_heat.axis("off")
    return im

# ---------------------------------------------------------------------------------------
# 3. OVERVIEW GRIDS -- more samples, fixed color scale, ground truth + prediction
# ---------------------------------------------------------------------------------------
for activity_name in ACTIVITY_PRIORITY:
    if activity_name not in SUMMARY_DF.index:
        print(f"[WARN] '{activity_name}' not found in patchcore_summary.csv -- skipping.")
        continue

    act_root = os.path.join(PATCHCORE_READY_ROOT, activity_name)
    threshold = float(SUMMARY_DF.loc[activity_name, "threshold"])
    calib_min = float(SUMMARY_DF.loc[activity_name, "calib_score_min"])
    calib_max = float(SUMMARY_DF.loc[activity_name, "calib_score_max"])

    try:
        memory_bank_path = find_memory_bank(activity_name)
    except FileNotFoundError as e:
        print(f"[WARN] {e}")
        continue

    print(f"\n{'='*70}\n{activity_name}  (threshold={threshold:.4f})\n{'='*70}")
    memory_bank = torch.load(memory_bank_path, map_location=device)

    normal_files = glob.glob(os.path.join(act_root, "test", "good", "*.png"))
    fall_files = glob.glob(os.path.join(act_root, "test", "anomaly", "*.png"))
    if len(normal_files) == 0 or len(fall_files) == 0:
        print(f"[WARN] Missing test images for {activity_name} -- skipping.")
        continue

    rng = random.Random(SEED)
    normal_sample = rng.sample(normal_files, min(N_EXAMPLES_PER_CLASS, len(normal_files)))
    fall_sample = rng.sample(fall_files, min(N_EXAMPLES_PER_CLASS, len(fall_files)))

    # Compute all results first (needed to fix the shared color scale before plotting -- FIX 1)
    results = []
    for fp in normal_sample:
        img_r, heat, score, peak_rc, peak_q = compute_anomaly_heatmap(fp, memory_bank)
        results.append(("Normal", fp, img_r, heat, score, peak_rc, peak_q))
    for fp in fall_sample:
        img_r, heat, score, peak_rc, peak_q = compute_anomaly_heatmap(fp, memory_bank)
        results.append(("Fall", fp, img_r, heat, score, peak_rc, peak_q))

    # Shared color scale across the WHOLE comparison set -- using the real calibration
    # range, not just this sample's own min/max, so it's consistent with the trained model
    vmin, vmax = calib_min, calib_max

    n_rows = len(results)
    fig, axes = plt.subplots(n_rows, 2, figsize=(9, 3.6 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for row, (gt, fp, img_r, heat, score, peak_rc, peak_q) in enumerate(results):
        norm_score = normalize_score(score, calib_min, calib_max)
        predicted = "FALL" if score >= threshold else "NORMAL"
        im = plot_one_result(axes[row, 0], axes[row, 1], img_r, heat, score, norm_score,
                              peak_rc, peak_q, gt, predicted, threshold, vmin, vmax)

    fig.suptitle(f"{activity_name}: Overview (shared color scale, threshold={threshold:.3f})",
                 fontsize=13, y=0.995)
    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02, label="Raw anomaly score")
    plt.savefig(os.path.join(OUTPUT_DIR, f"heatmap_{activity_name}.png"), dpi=140, bbox_inches="tight")
    plt.close(fig)
    print(f"[INFO] Saved overview grid: heatmap_{activity_name}.png")

# ---------------------------------------------------------------------------------------
# 4. THE 15-IMAGE INDIVIDUAL TEST SUITE (5 good + 5 prefall_context + 5 anomaly)
# ---------------------------------------------------------------------------------------
individual_summary_rows = []

for activity_name in ACTIVITY_PRIORITY:
    if activity_name not in SUMMARY_DF.index:
        continue
    act_root = os.path.join(PATCHCORE_READY_ROOT, activity_name)
    threshold = float(SUMMARY_DF.loc[activity_name, "threshold"])
    calib_min = float(SUMMARY_DF.loc[activity_name, "calib_score_min"])
    calib_max = float(SUMMARY_DF.loc[activity_name, "calib_score_max"])

    try:
        memory_bank_path = find_memory_bank(activity_name)
    except FileNotFoundError:
        continue
    memory_bank = torch.load(memory_bank_path, map_location=device)

    categories = {
        "good": ("Normal", os.path.join(act_root, "test", "good")),
        "prefall_context": ("Normal", os.path.join(act_root, "test", "prefall_context")),
        "anomaly": ("Fall", os.path.join(act_root, "test", "anomaly")),
    }

    rng = random.Random(SEED + 1)
    print(f"\n{'='*70}\n{activity_name} -- 15-image individual test suite\n{'='*70}")

    for category, (ground_truth, folder) in categories.items():
        files = glob.glob(os.path.join(folder, "*.png"))
        if len(files) == 0:
            print(f"[WARN] No images found in {folder} -- skipping this category for {activity_name}.")
            continue
        sample = rng.sample(files, min(N_INDIVIDUAL_TEST_PER_CATEGORY, len(files)))

        out_dir = os.path.join(INDIVIDUAL_DIR, activity_name, category)
        os.makedirs(out_dir, exist_ok=True)

        for fp in sample:
            img_r, heat, score, peak_rc, peak_q = compute_anomaly_heatmap(fp, memory_bank)
            norm_score = normalize_score(score, calib_min, calib_max)
            predicted = "FALL" if score >= threshold else "NORMAL"
            correct = (ground_truth == "Fall" and predicted == "FALL") or \
                      (ground_truth == "Normal" and predicted == "NORMAL")

            fig, (ax_img, ax_heat) = plt.subplots(1, 2, figsize=(10, 5))
            im = plot_one_result(ax_img, ax_heat, img_r, heat, score, norm_score, peak_rc,
                                  peak_q, ground_truth, predicted, threshold, calib_min, calib_max)
            fname = os.path.basename(fp)
            fig.suptitle(f"{activity_name} | {category} | {fname}", fontsize=10, y=1.02)
            plt.tight_layout()
            out_path = os.path.join(out_dir, f"{os.path.splitext(fname)[0]}_result.png")
            plt.savefig(out_path, dpi=130, bbox_inches="tight")
            plt.close(fig)

            individual_summary_rows.append({
                "activity": activity_name,
                "category": category,
                "filename": fname,
                "ground_truth": ground_truth,
                "predicted": predicted,
                "correct": correct,
                "raw_score": score,
                "normalized_score": norm_score,
                "threshold": threshold,
                "peak_quadrant": peak_q,
            })
            print(f"  [{category}] {fname}: GT={ground_truth} Pred={predicted} "
                  f"({'OK' if correct else 'MISMATCH'}) score={score:.3f} peak={peak_q}")

individual_summary_df = pd.DataFrame(individual_summary_rows)
individual_summary_df.to_csv(os.path.join(INDIVIDUAL_DIR, "individual_test_summary.csv"), index=False)

print(f"\n{'='*70}\nINDIVIDUAL TEST SUITE SUMMARY\n{'='*70}")
if len(individual_summary_df) > 0:
    print(individual_summary_df.groupby(["activity", "category"])["correct"].agg(["sum", "count"]).to_string())
    print(f"\n[INFO] Mismatches (worth inspecting first):")
    mismatches = individual_summary_df[~individual_summary_df["correct"]]
    print(mismatches[["activity", "category", "filename", "ground_truth", "predicted", "raw_score", "peak_quadrant"]].to_string(index=False)
          if len(mismatches) > 0 else "  None -- all 15-per-activity predictions matched ground truth.")

print(f"\n[DONE] All outputs saved under {OUTPUT_DIR}:")
print(f"  - heatmap_{{Activity}}.png          (overview grids, fixed color scale)")
print(f"  - individual_test_15/{{Activity}}/{{category}}/*.png  (one file per test image)")
print(f"  - individual_test_15/individual_test_summary.csv       (all results in one table)")
print("\n[HOW TO READ] White circle = the single hottest patch (what drove the score).")
print("              For genuine FALL predictions, check the 'Peak region' label -- it")
print("              should usually say Kurtogram, not Scalogram, if the model is keying")
print("              off the impact signature it was designed around.")

[INFO] Device: cuda
[INFO] Loading wide_resnet50_2 pretrained weights (attempt 1/3)...
[INFO] Pretrained weights loaded successfully.

Walking  (threshold=2.3569)
[INFO] Saved overview grid: heatmap_Walking.png

Sitting  (threshold=2.3123)
[INFO] Saved overview grid: heatmap_Sitting.png

Standing_GettingUp  (threshold=2.4507)
[INFO] Saved overview grid: heatmap_Standing_GettingUp.png

Jogging  (threshold=2.3101)
[INFO] Saved overview grid: heatmap_Jogging.png

Walking -- 15-image individual test suite
  [good] D02_SE05_R01_T0.0-5.0s.png: GT=Normal Pred=NORMAL (OK) score=2.224 peak=Gyro-Kurtogram
  [good] D02_SA18_R01_T82.5-87.5s.png: GT=Normal Pred=NORMAL (OK) score=2.160 peak=Gyro-Kurtogram
  [good] D01_SA01_R01_T90.0-95.0s.png: GT=Normal Pred=NORMAL (OK) score=2.175 peak=Acc1-Scalogram
  [good] D02_SA18_R01_T7.5-12.5s.png: GT=Normal Pred=NORMAL (OK) score=2.166 peak=Gyro-Kurtogram
  [good] D01_SE04_R01_T77.5-82.5s.png: GT=Normal Pred=NORMAL (OK) score=2.302 peak=Gyro-Scalogram
  [pre

In [6]:
"""Manual verification tool · PY
===================================================================================================
MANUAL VERIFICATION TOOL -- spot-check the automatic window-selection heuristic
===================================================================================================
Run this using your ORIGINAL Mosaic_PatchCore_2x2 folder (before refine_anomaly_windows.py
moved anything) -- so point PATCHCORE_ROOT at your raw generated dataset, not PatchCore_Ready.

WHAT THIS DOES:
For a random sample of fall trials, shows ALL of that trial's windows side-by-side, in time
order, each labeled with its kurtogram intensity score and whether the refiner kept it as
"anomaly", routed it to "prefall_context", or excluded it. This lets you visually verify: does
the window the heuristic picked as "the impact" actually LOOK like it has a sharp, localized
spike -- or did the heuristic pick wrong?

HOW TO USE THE OUTPUT:
Look at each row (one fall trial). The window(s) labeled "KEPT (anomaly)" should visibly show
a bright, tight, high-contrast blob in the kurtogram cell (right column) that the other windows
in the same row don't have. If a "KEPT" window looks just as calm as its neighbors, or an
"EXCLUDED"/"PREFALL" window actually looks more dramatic, that's a sign TOP_K_WINDOWS_PER_TRIAL
or PREFALL_BUFFER_WINDOWS (in refine_anomaly_windows.py) may need adjusting for that activity.
===================================================================================================
"""

import os
import re
import glob
import random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

# ---------------------------------------------------------------------------------------
# 0. CONFIG -- must match refine_anomaly_windows.py's settings
# ---------------------------------------------------------------------------------------
SEED = 42
N_TRIALS_TO_INSPECT = 8     # per activity -- increased from 5 for more thorough spot-checking
TOP_K_WINDOWS_PER_TRIAL = 2
PREFALL_BUFFER_WINDOWS = 1

CELL_SIZE = 224
GUTTER_PX = 8
KURTOGRAM_X0 = CELL_SIZE + GUTTER_PX
KURTOGRAM_X1 = KURTOGRAM_X0 + CELL_SIZE

ACTIVITY_GROUPS = ["Walking", "Jogging", "Sitting", "Standing_GettingUp"]
FALL_CODE_TO_ACTIVITY = {
    "F01": "Walking", "F02": "Walking", "F03": "Walking", "F04": "Walking",
    "F06": "Walking", "F07": "Walking", "F05": "Jogging",
    "F08": "Standing_GettingUp", "F09": "Standing_GettingUp",
    "F10": "Sitting", "F11": "Sitting", "F12": "Sitting",
    "F13": "Sitting", "F14": "Sitting", "F15": "Sitting",
}

def find_mosaic_patchcore_root():
    candidates = glob.glob("/kaggle/input/**/Mosaic_PatchCore_2x2", recursive=True)
    if not candidates:
        raise FileNotFoundError("Could not find 'Mosaic_PatchCore_2x2' under /kaggle/input.")
    return candidates[0]

PATCHCORE_ROOT = find_mosaic_patchcore_root()
OUTPUT_DIR = "/kaggle/working/manual_verification"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def kurtogram_intensity(filepath):
    img = Image.open(filepath).convert("RGB")
    arr = np.array(img)
    if arr.shape[0] < KURTOGRAM_X1 or arr.shape[1] < KURTOGRAM_X1:
        w = arr.shape[1]
        kurt_region = arr[:, w // 2:]
    else:
        kurt_region = arr[:, KURTOGRAM_X0:KURTOGRAM_X1]
    return float(kurt_region.std())

def trial_id_from_filename(filepath):
    stem = os.path.splitext(os.path.basename(filepath))[0]
    return re.sub(r"_T[\d.]+-[\d.]+s$", "", stem)

def window_start_time(filepath):
    stem = os.path.splitext(os.path.basename(filepath))[0]
    m = re.search(r"_T([\d.]+)-[\d.]+s$", stem)
    return float(m.group(1)) if m else None

def classify_windows_in_trial(filepaths):
    """Reproduces refine_anomaly_windows.py's exact classification logic for one trial,
    so this tool's labels match what the refiner actually decided."""
    items = [{"filepath": fp, "t0": window_start_time(fp),
              "kurt_intensity": kurtogram_intensity(fp)} for fp in filepaths]
    items.sort(key=lambda x: (x["t0"] if x["t0"] is not None else 0))
    for i, item in enumerate(items):
        item["position"] = i

    sorted_by_intensity = sorted(items, key=lambda x: -x["kurt_intensity"])
    top_k = sorted_by_intensity[:min(TOP_K_WINDOWS_PER_TRIAL, len(items))]
    top_k_positions = {item["position"] for item in top_k}

    if top_k_positions:
        earliest_impact_pos = min(top_k_positions)
        for item in items:
            if item["position"] in top_k_positions:
                item["label"] = "KEPT (anomaly)"
            elif item["position"] < earliest_impact_pos - PREFALL_BUFFER_WINDOWS:
                item["label"] = "PREFALL_CONTEXT"
            else:
                item["label"] = "EXCLUDED"
    return items

# ---------------------------------------------------------------------------------------
# 1. FOR EACH ACTIVITY, SAMPLE N TRIALS AND VISUALIZE ALL THEIR WINDOWS SIDE-BY-SIDE
# ---------------------------------------------------------------------------------------
rng = random.Random(SEED)

for activity_name in ACTIVITY_GROUPS:
    fall_dir = os.path.join(PATCHCORE_ROOT, "Fall")
    if not os.path.isdir(fall_dir):
        print(f"[WARN] No Fall folder found at {fall_dir}")
        continue

    # Collect all fall images belonging to this activity group
    all_fall_files = glob.glob(os.path.join(fall_dir, "**", "*.png"), recursive=True)
    activity_files = []
    for fp in all_fall_files:
        fname = os.path.basename(fp)
        m = re.match(r"^(F\d{2})_", fname)
        if m and FALL_CODE_TO_ACTIVITY.get(m.group(1)) == activity_name:
            activity_files.append(fp)

    if not activity_files:
        print(f"[WARN] No fall images found for {activity_name} -- skipping.")
        continue

    # Group into trials
    trials = {}
    for fp in activity_files:
        tid = trial_id_from_filename(fp)
        trials.setdefault(tid, []).append(fp)

    trial_ids = list(trials.keys())
    rng.shuffle(trial_ids)
    sample_trial_ids = trial_ids[:min(N_TRIALS_TO_INSPECT, len(trial_ids))]

    print(f"\n{'='*70}\n{activity_name} -- inspecting {len(sample_trial_ids)} trials\n{'='*70}")

    for trial_id in sample_trial_ids:
        items = classify_windows_in_trial(trials[trial_id])
        n_windows = len(items)

        fig, axes = plt.subplots(1, n_windows, figsize=(4.5 * n_windows, 5.8))
        if n_windows == 1:
            axes = [axes]

        for ax, item in zip(axes, items):
            img = Image.open(item["filepath"]).convert("RGB")
            ax.imshow(img)
            color = {"KEPT (anomaly)": "red", "PREFALL_CONTEXT": "green", "EXCLUDED": "gray"}[item["label"]]
            ax.set_title(f"{item['label']}\nkurt_intensity={item['kurt_intensity']:.1f}",
                         color=color, fontsize=10, fontweight="bold", pad=10)
            ax.axis("off")
            # Highlight the kurtogram cell region with a red box for easy visual reference
            rect_x, rect_y = KURTOGRAM_X0, 0
            ax.add_patch(plt.Rectangle((rect_x, rect_y), CELL_SIZE, CELL_SIZE * 2 + GUTTER_PX,
                                         fill=False, edgecolor="cyan", linewidth=1.5))

        # FIX: reserve explicit top margin for the suptitle so it never overlaps the
        # per-window titles below it -- rect=[left, bottom, right, top] leaves the top 8%
        # of the figure clear for the suptitle text.
        plt.suptitle(f"{activity_name} | Trial: {trial_id} | {n_windows} windows, time-ordered left-to-right",
                     fontsize=12, y=0.99)
        plt.tight_layout(rect=[0, 0, 1, 0.93])
        safe_trial_id = trial_id.replace("/", "_").replace("\\", "_")
        out_path = os.path.join(OUTPUT_DIR, f"{activity_name}_{safe_trial_id}.png")
        plt.savefig(out_path, dpi=130, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved: {out_path}")

print(f"\n[DONE] Manual verification images saved under {OUTPUT_DIR}")
print("[HOW TO USE] Open a few of these images. Cyan box = kurtogram cell region (both rows).")
print("             Red title = kept as anomaly, Green = prefall_context, Gray = excluded.")
print("             Confirm: does the RED-titled window actually show a visibly brighter/more")
print("             colorful blob inside the cyan box compared to the other windows in that")
print("             same row? If yes for most trials you check, the heuristic is working.")
print("             If not, consider adjusting TOP_K_WINDOWS_PER_TRIAL in refine_anomaly_windows.py.")


Walking -- inspecting 8 trials
  Saved: /kaggle/working/manual_verification/Walking_F04_SA20_R03.png
  Saved: /kaggle/working/manual_verification/Walking_F01_SA20_R01.png
  Saved: /kaggle/working/manual_verification/Walking_F06_SA14_R03.png
  Saved: /kaggle/working/manual_verification/Walking_F06_SA04_R03.png
  Saved: /kaggle/working/manual_verification/Walking_F01_SA13_R02.png
  Saved: /kaggle/working/manual_verification/Walking_F07_SA03_R04.png
  Saved: /kaggle/working/manual_verification/Walking_F06_SA16_R05.png
  Saved: /kaggle/working/manual_verification/Walking_F03_SA15_R02.png

Jogging -- inspecting 8 trials
  Saved: /kaggle/working/manual_verification/Jogging_F05_SA02_R02.png
  Saved: /kaggle/working/manual_verification/Jogging_F05_SA12_R03.png
  Saved: /kaggle/working/manual_verification/Jogging_F05_SA01_R04.png
  Saved: /kaggle/working/manual_verification/Jogging_F05_SA13_R05.png
  Saved: /kaggle/working/manual_verification/Jogging_F05_SA16_R02.png
  Saved: /kaggle/working/m